# Generic Web Page Parser Test

Platform: any URL (blogs, news, docs, etc.)

Strategy: trafilatura (best-in-class content extraction) -> Markdown

Fallback: requests + readability-like heuristic

In [ ]:
TEST_URL = "https://example.com/article"  # Change to any URL
import os
OUTPUT_DIR = os.path.join(os.getcwd(), "test-output")

In [ ]:
import os
for pkg, cmd in [("trafilatura", "pip install trafilatura"),
                 ("requests", "pip install requests"),
                 ("bs4", "pip install beautifulsoup4")]:
    try:
        __import__(pkg)
        print(f"[OK] {pkg}")
    except ImportError:
        print(f"[MISSING] {pkg} -> {cmd}")
        raise SystemExit("Install missing packages first.")

In [ ]:
import trafilatura

# Method 1: trafilatura (recommended)
downloaded = trafilatura.fetch_url(TEST_URL)

if downloaded:
    result = trafilatura.extract(
        downloaded,
        output_format="markdown",
        include_comments=False,
        include_images=True,
        include_tables=True,
    )
    title = trafilatura.extract_metadata(downloaded).get("title", "") if trafilatura.extract_metadata(downloaded) else ""
else:
    result = None
    title = ""

print(f"Fetched: {len(downloaded) if downloaded else 0} chars")
print(f"Title: {title}")
print(f"Markdown: {len(result) if result else 0} chars")

In [ ]:
# Fallback: requests + simple extraction if trafilatura fails
if not result:
    import requests
    from bs4 import BeautifulSoup

    resp = requests.get(TEST_URL, headers={"User-Agent": "Mozilla/5.0"}, timeout=15)
    soup = BeautifulSoup(resp.text, "html.parser")
    title = soup.title.get_text(strip=True) if soup.title else ""

    # Remove noise
    for tag in soup.find_all(["script", "style", "nav", "header", "footer", "aside"]):
        tag.decompose()

    body = soup.find("article") or soup.find("main") or soup.find("body")
    if body:
        from markdownify import markdownify as md
        result = md(str(body), heading_style="ATX").strip()
    else:
        result = ""

    downloaded = resp.text
    print(f"Fallback used. Markdown: {len(result)} chars")

In [ ]:
from IPython.display import Markdown as IPMarkdown, display
if result:
    preview = result[:4000] + ("... (truncated)" if len(result) > 4000 else "")
    display(IPMarkdown(preview))
else:
    print("No content extracted.")

In [ ]:
import os
from urllib.parse import urlparse

os.makedirs(OUTPUT_DIR, exist_ok=True)
slug = urlparse(TEST_URL).path.strip("/").replace("/", "-") or "page"

if downloaded:
    with open(os.path.join(OUTPUT_DIR, f"{slug}.html"), "w", encoding="utf-8") as f:
        f.write(downloaded)
if result:
    with open(os.path.join(OUTPUT_DIR, f"{slug}.md"), "w", encoding="utf-8") as f:
        f.write(result)
print("Saved to test-output/")